In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("CSV_to_Parquet_Weather_SP")
    .getOrCreate()
)

In [ ]:
csv_path = "/home/jovyan/work/data/raw/*.csv"
parquet_path = "/home/jovyan/work/data/processed/weather_sp_parquet"

df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .csv(csv_path)
)

In [ ]:
df_csv.show(5)
df_csv.printSchema()

# Veja primeiro os estados existentes
df_csv.select("state").distinct().show()

In [ ]:
# Filtra somente SP
df_sp = df_csv.filter(col("state") == "SP")

df_sp.show(5)
df_sp.printSchema()

In [ ]:
# Salva Parquet já reduzido
df_sp.write.mode("overwrite").parquet(parquet_path)

print(f"Arquivo Parquet de SP salvo em: {parquet_path}")

In [ ]:
df_parquet = spark.read.parquet(parquet_path)

df_parquet.show(5)
df_parquet.printSchema()

linhas_sp = df_sp.count()
linhas_parquet = df_parquet.count()

print(f"Linhas no dataframe SP: {linhas_sp}")
print(f"Linhas no Parquet: {linhas_parquet}")

if linhas_sp == linhas_parquet:
    print("Conversão realizada com sucesso. A quantidade de linhas de SP foi preservada.")
else:
    print("Atenção: a quantidade de linhas ficou diferente após a conversão.")